Data Quality

Lectura de bronce

In [0]:
# ============================================
# CONFIGURACIÓN DATA QUALITY
# ============================================

from pyspark.sql import functions as F

tabla_bronze = "workspace.default.bronze_nyctaxi"

df_bronze = spark.table(tabla_bronze)

print(f"Registros en Bronze: {df_bronze.count()}")

Definicion de reglas

In [0]:
# ============================================
# REGLAS DE CALIDAD
# ============================================

regla_pickup_nulo = F.col("tpep_pickup_datetime").isNull()

regla_dropoff_nulo = F.col("tpep_dropoff_datetime").isNull()

regla_distancia_negativa = F.col("trip_distance") < 0

regla_tarifa_negativa = F.col("fare_amount") < 0

regla_fecha_invalida = (
    F.col("tpep_dropoff_datetime")
    < F.col("tpep_pickup_datetime")
)

regla_error = (
    regla_pickup_nulo
    | regla_dropoff_nulo
    | regla_distancia_negativa
    | regla_tarifa_negativa
    | regla_fecha_invalida
)

Conteo de errores

In [0]:
# ============================================
# MEDICIÓN DE ERRORES DE CALIDAD
# ============================================

total_registros = df_bronze.count()

registros_con_error = (
    df_bronze
    .filter(regla_error)
    .count()
)

registros_validos = total_registros - registros_con_error

print(f"Total registros: {total_registros}")
print(f"Registros válidos: {registros_validos}")
print(f"Registros con errores: {registros_con_error}")

Identificacion de errores individuales

In [0]:
# ============================================
# DETALLE DE ERRORES POR REGLA
# ============================================

errores_por_regla = {
    "pickup_nulo": df_bronze.filter(regla_pickup_nulo).count(),
    "dropoff_nulo": df_bronze.filter(regla_dropoff_nulo).count(),
    "distancia_negativa": df_bronze.filter(regla_distancia_negativa).count(),
    "tarifa_negativa": df_bronze.filter(regla_tarifa_negativa).count(),
    "fecha_invalida": df_bronze.filter(regla_fecha_invalida).count()
}

for regla, cantidad in errores_por_regla.items():
    print(f"{regla}: {cantidad}")

Separar registros validos e invalidos

In [0]:
# ============================================
# SEPARAR REGISTROS VÁLIDOS E INVÁLIDOS
# ============================================

df_validos = (
    df_bronze
    .filter(~regla_error)
)

df_invalidos = (
    df_bronze
    .filter(regla_error)
)

print(f"Registros válidos: {df_validos.count()}")
print(f"Registros inválidos: {df_invalidos.count()}")

Creacion de table cuarentena

In [0]:
# ============================================
# CREAR TABLA DE CUARENTENA
# ============================================

(
    df_invalidos
    .write
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.quarantine_nyctaxi"
    )
)

print("Tabla de cuarentena creada correctamente.")

In [0]:
%sql

SELECT COUNT(*) AS registros_cuarentena
FROM workspace.default.quarantine_nyctaxi;

CREACION DE SILVER

In [0]:
# ============================================
# CREAR TABLA SILVER
# ============================================

(
    df_validos
    .write
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.silver_nyctaxi"
    )
)

print("Tabla Silver creada correctamente.")

In [0]:
%sql

SELECT COUNT(*) AS registros_silver
FROM workspace.default.silver_nyctaxi;

Identificar duplicados de SILVER

In [0]:
# ============================================
# DETECCIÓN DE DUPLICADOS
# ============================================

total_silver = df_validos.count()

total_hash_unicos = (
    df_validos
    .select("_hash_registro")
    .distinct()
    .count()
)

registros_duplicados = total_silver - total_hash_unicos

print(f"Registros Silver: {total_silver}")
print(f"Registros únicos: {total_hash_unicos}")
print(f"Registros duplicados: {registros_duplicados}")

Eliminar duplicadosde SILVER aplicando dropDuplicates sobre _hash_registro

In [0]:
# ============================================
# DEDUPLICAR REGISTROS VÁLIDOS
# ============================================

df_silver = (
    df_validos
    .dropDuplicates(["_hash_registro"])
)

print(f"Registros antes de deduplicar: {df_validos.count()}")
print(f"Registros después de deduplicar: {df_silver.count()}")

SILVER depurado

In [0]:
# ============================================
# GUARDAR SILVER DEFINITIVA
# ============================================

(
    df_silver
    .write
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.silver_nyctaxi"
    )
)

print("Tabla Silver actualizada correctamente.")

In [0]:
%sql

SELECT COUNT(*) AS registros_silver
FROM workspace.default.silver_nyctaxi;

Actualizar tabla de control

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

tabla_bronze = "workspace.default.bronze_nyctaxi"

df_bronze_control = spark.table(tabla_bronze)

regla_error = (
    F.col("tpep_pickup_datetime").isNull()
    | F.col("tpep_dropoff_datetime").isNull()
    | (F.col("trip_distance") < 0)
    | (F.col("fare_amount") < 0)
    | (
        F.col("tpep_dropoff_datetime")
        < F.col("tpep_pickup_datetime")
    )
)

df_validos_control = (
    df_bronze_control
    .filter(~regla_error)
)

print(f"Registros válidos: {df_validos_control.count()}")

In [0]:
ventana_duplicados = (
    Window
    .partitionBy("_hash_registro")
    .orderBy(
        F.col("_fecha_ingesta").asc(),
        F.col("_archivo_origen").asc()
    )
)

df_clasificado_control = (
    df_validos_control
    .withColumn(
        "_orden_duplicado",
        F.row_number().over(ventana_duplicados)
    )
)

df_unicos_control = (
    df_clasificado_control
    .filter(F.col("_orden_duplicado") == 1)
)

df_duplicados_control = (
    df_clasificado_control
    .filter(F.col("_orden_duplicado") > 1)
)

print(f"Registros únicos: {df_unicos_control.count()}")
print(f"Registros duplicados: {df_duplicados_control.count()}")

Discriminado reegistros duplicados y validos por cada carga

In [0]:
resumen_por_carga = (
    df_clasificado_control
    .withColumn(
        "_tipo_registro",
        F.when(
            F.col("_orden_duplicado") == 1,
            "UNICO"
        ).otherwise("DUPLICADO")
    )
    .groupBy("_archivo_origen")
    .pivot("_tipo_registro", ["UNICO", "DUPLICADO"])
    .count()
    .fillna(0)
    .orderBy("_archivo_origen")
)

display(resumen_por_carga)

In [0]:
from delta.tables import DeltaTable

tabla_control = "workspace.default.control_cargas_nyctaxi"

df_metricas = (
    df_clasificado_control
    .withColumn(
        "_es_unico",
        F.when(F.col("_orden_duplicado") == 1, 1).otherwise(0)
    )
    .withColumn(
        "_es_duplicado",
        F.when(F.col("_orden_duplicado") > 1, 1).otherwise(0)
    )
    .groupBy("_archivo_origen")
    .agg(
        F.sum("_es_unico").cast("long").alias("_registros_nuevos"),
        F.sum("_es_duplicado").cast("long").alias("_registros_duplicados")
    )
)

df_rechazados = (
    df_bronze_control
    .filter(regla_error)
    .groupBy("_archivo_origen")
    .agg(
        F.count("*").cast("long").alias("_registros_rechazados")
    )
)

df_metricas_finales = (
    df_metricas
    .join(
        df_rechazados,
        on="_archivo_origen",
        how="left"
    )
    .fillna(0, subset=["_registros_rechazados"])
)

display(
    df_metricas_finales
    .orderBy("_archivo_origen")
)

In [0]:
from delta.tables import DeltaTable

tabla_control = "workspace.default.control_cargas_nyctaxi"

tabla_delta = DeltaTable.forName(
    spark,
    tabla_control
)

(
    tabla_delta.alias("control")
    .merge(
        df_metricas_finales.alias("metricas"),
        "control._archivo_origen = metricas._archivo_origen"
    )
    .whenMatchedUpdate(
        set={
            "_registros_nuevos": "metricas._registros_nuevos",
            "_registros_duplicados": "metricas._registros_duplicados",
            "_registros_rechazados": "metricas._registros_rechazados",
            "_estado": "CASE WHEN " +
                       "metricas._registros_nuevos + " +
                       "metricas._registros_duplicados + " +
                       "metricas._registros_rechazados = " +
                       "control._registros_leidos " +
                       "THEN 'OK' ELSE 'ERROR' END"
        }
    )
    .execute()
)

print("Tabla de control actualizada correctamente.")

verificacion

In [0]:
%sql

SELECT
    _archivo_origen,
    _registros_leidos,
    _registros_nuevos,
    _registros_duplicados,
    _registros_rechazados,
    _estado
FROM workspace.default.control_cargas_nyctaxi
ORDER BY _archivo_origen;